## 井旁道子波增益 QC

使用`wavelet_amplitude_gain_attribute_apply_random_traces@20260429.ipynb`中的动态分段均方根/增益拟合方法，预测每口井旁道上的增益曲线，然后比较固定尺度合成增益与均方根预测增益的合成结果。


In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

repo_root = Path.cwd().resolve()
if repo_root.name == "notebooks":
    repo_root = repo_root.parent
if not (repo_root / "src").exists():
    raise RuntimeError("Could not locate repo root containing 'src'.")

src_root = repo_root / "src"
if str(src_root) not in sys.path:
    sys.path.insert(0, str(src_root))

plt.rcParams["figure.dpi"] = 120
pd.set_option("display.max_columns", 120)


In [ ]:
data_root = repo_root / "data"

wavelet_batch_output_dir = data_root / "output_wavelet_batch_synthetic_depth_20260428"
batch_metrics_file = wavelet_batch_output_dir / "wavelet_batch_metrics.csv"

output_dir = data_root / "output_wavelet_gain_wellside_synthetic_qc_20260429"
well_qc_dir = output_dir / "well_qc"
figure_dir = output_dir / "figures"
training_samples_file = output_dir / "wellside_gain_training_samples.csv"
fit_metrics_file = output_dir / "wellside_gain_fit_metrics.csv"
well_qc_dir.mkdir(parents=True, exist_ok=True)
figure_dir.mkdir(parents=True, exist_ok=True)

# Adaptive segmentation parameters for well-side training samples.
min_segment_valid_samples = 8
max_segment_count = 20
min_segments_per_well = 1
gain_eps = 1e-12

selected_attribute = "seismic_rms"
min_batch_corr = None
application_rms_window_m = None
prediction_clip_percentiles = (5.0, 95.0)
attribute_floor_fraction = 0.10

prediction_gain_column = "gain_pred_clipped"
required_paths = [batch_metrics_file]
for path in required_paths:
    if not path.exists():
        raise FileNotFoundError(path)

print("Batch metrics:", batch_metrics_file)
print("Output dir:", output_dir)


In [ ]:
def resolve_artifact_path(path_value: str | Path) -> Path:
    path = Path(path_value)
    return path if path.is_absolute() else repo_root / path


def safe_name(well_name: str) -> str:
    return well_name.replace("/", "_").replace("\\", "_").replace(" ", "_")


def centered_moving_sum(values: np.ndarray, window_samples: int) -> np.ndarray:
    values = np.asarray(values, dtype=float)
    window_samples = int(max(window_samples, 1))
    if values.size == 0:
        return values.copy()
    radius_left = window_samples // 2
    radius_right = window_samples - 1 - radius_left
    padded = np.pad(values, (radius_left, radius_right), mode="constant", constant_values=0.0)
    cumsum = np.concatenate([[0.0], np.cumsum(padded)])
    return cumsum[window_samples:] - cumsum[:-window_samples]


def centered_moving_rms(values: np.ndarray, window_samples: int) -> np.ndarray:
    values = np.asarray(values, dtype=float)
    valid = np.isfinite(values)
    numerator = centered_moving_sum(np.where(valid, values**2, 0.0), window_samples)
    denominator = centered_moving_sum(valid.astype(float), window_samples)
    out = np.full(values.shape, np.nan, dtype=float)
    positive = denominator > 0.0
    out[positive] = np.sqrt(numerator[positive] / denominator[positive])
    return out


def resolve_odd_window_samples(window_m: float, depth_values: np.ndarray) -> tuple[int, float]:
    depth_values = np.asarray(depth_values, dtype=float)
    valid_depth = depth_values[np.isfinite(depth_values)]
    if valid_depth.size < 3:
        raise ValueError("Need at least three finite depth samples to resolve RMS window.")
    depth_step_m = float(np.nanmedian(np.abs(np.diff(valid_depth))))
    if not np.isfinite(depth_step_m) or depth_step_m <= 0.0:
        raise ValueError(f"Invalid inferred depth sample step: {depth_step_m}")
    window_samples = int(round(float(window_m) / depth_step_m))
    window_samples = max(window_samples, 3)
    if window_samples % 2 == 0:
        window_samples += 1
    return window_samples, depth_step_m


def finite_pearson(x: np.ndarray, y: np.ndarray, mask: np.ndarray | None = None) -> float:
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    valid = np.isfinite(x) & np.isfinite(y)
    if mask is not None:
        valid &= np.asarray(mask, dtype=bool)
    if int(valid.sum()) < 5:
        return np.nan
    if np.nanstd(x[valid]) <= 0.0 or np.nanstd(y[valid]) <= 0.0:
        return np.nan
    return float(np.corrcoef(x[valid], y[valid])[0, 1])


def fit_log_linear(x_log: np.ndarray, y_log: np.ndarray) -> dict:
    x_log = np.asarray(x_log, dtype=float)
    y_log = np.asarray(y_log, dtype=float)
    mask = np.isfinite(x_log) & np.isfinite(y_log)
    if int(mask.sum()) < 5:
        raise ValueError("Need at least five finite samples for log-linear fitting.")
    x = x_log[mask]
    y = y_log[mask]
    design = np.column_stack([np.ones_like(x), x])
    intercept, slope = np.linalg.lstsq(design, y, rcond=None)[0]
    pred = intercept + slope * x
    ss_res = float(np.sum((y - pred) ** 2))
    ss_tot = float(np.sum((y - np.mean(y)) ** 2))
    r2 = np.nan if ss_tot <= 0.0 else 1.0 - ss_res / ss_tot
    return {
        "intercept": float(intercept),
        "slope": float(slope),
        "n_samples": int(mask.sum()),
        "pearson": finite_pearson(x, y),
        "r2_in_sample": float(r2),
        "mae_log_gain": float(np.mean(np.abs(y - pred))),
        "rmse_log_gain": float(np.sqrt(np.mean((y - pred) ** 2))),
    }


def split_valid_indices(
    valid_indices: np.ndarray,
    *,
    min_valid_samples: int,
    max_segments: int,
    min_segments: int,
) -> list[np.ndarray]:
    valid_indices = np.asarray(valid_indices, dtype=np.int64)
    if valid_indices.size < int(min_valid_samples):
        return []
    segment_count = int(min(max_segments, valid_indices.size // int(min_valid_samples)))
    segment_count = max(int(min_segments), segment_count)
    return [segment for segment in np.array_split(valid_indices, segment_count) if segment.size >= min_valid_samples]


def positive_ls_gain(
    seismic_values: np.ndarray,
    synthetic_raw_values: np.ndarray,
    *,
    eps: float,
    min_valid_samples: int,
) -> float:
    seismic_values = np.asarray(seismic_values, dtype=float)
    synthetic_raw_values = np.asarray(synthetic_raw_values, dtype=float)
    valid = np.isfinite(seismic_values) & np.isfinite(synthetic_raw_values)
    if int(valid.sum()) < int(min_valid_samples):
        return np.nan
    numerator = float(np.sum(seismic_values[valid] * synthetic_raw_values[valid]))
    denominator = float(np.sum(synthetic_raw_values[valid] ** 2))
    gain = numerator / (denominator + float(eps) * max(int(valid.sum()), 1))
    return float(gain) if np.isfinite(gain) and gain > 0.0 else np.nan


def segment_attribute_values(seismic_values: np.ndarray) -> dict:
    seismic_values = np.asarray(seismic_values, dtype=float)
    finite = np.isfinite(seismic_values)
    if not np.any(finite):
        return {"seismic_rms": np.nan, "seismic_abs_mean": np.nan, "seismic_abs_p90": np.nan}
    values = seismic_values[finite]
    abs_values = np.abs(values)
    return {
        "seismic_rms": float(np.sqrt(np.nanmean(values**2))),
        "seismic_abs_mean": float(np.nanmean(abs_values)),
        "seismic_abs_p90": float(np.nanpercentile(abs_values, 90.0)),
    }


def set_log_column(df: pd.DataFrame, source_column: str, log_column: str) -> None:
    values = df[source_column].to_numpy(dtype=float)
    df[log_column] = np.nan
    positive = values > 0.0
    df.loc[positive, log_column] = np.log(values[positive])


def nmae(seismic_norm: np.ndarray, synthetic: np.ndarray, mask: np.ndarray) -> float:
    seismic_norm = np.asarray(seismic_norm, dtype=float)
    synthetic = np.asarray(synthetic, dtype=float)
    valid = np.asarray(mask, dtype=bool) & np.isfinite(seismic_norm) & np.isfinite(synthetic)
    if int(valid.sum()) < 5:
        return np.nan
    denominator = float(np.sum(np.abs(seismic_norm[valid])))
    if denominator <= 0.0:
        return np.nan
    return float(np.sum(np.abs(seismic_norm[valid] - synthetic[valid])) / denominator)


def candidate_metrics(seismic_norm: np.ndarray, synthetic: np.ndarray, mask: np.ndarray) -> dict:
    valid = np.asarray(mask, dtype=bool) & np.isfinite(seismic_norm) & np.isfinite(synthetic)
    return {
        "corr": finite_pearson(seismic_norm, synthetic, valid),
        "nmae": nmae(seismic_norm, synthetic, valid),
        "n_eval_samples": int(valid.sum()),
    }


def eval_mask_to_bool(series: pd.Series) -> np.ndarray:
    if series.dtype == bool:
        return series.to_numpy(dtype=bool)
    return series.astype(str).str.lower().isin(["true", "1", "yes"]).to_numpy(dtype=bool)


def predict_gain_from_trace_rms(
    seismic_norm: np.ndarray,
    *,
    window_samples: int,
    intercept: float,
    slope: float,
    attribute_floor: float,
    log_gain_clip: tuple[float, float],
) -> pd.DataFrame:
    seismic_norm = np.asarray(seismic_norm, dtype=float)
    seismic_rms = centered_moving_rms(seismic_norm, window_samples)
    rms_safe = np.maximum(seismic_rms, float(attribute_floor))
    log_seismic_rms = np.where(np.isfinite(rms_safe) & (rms_safe > 0.0), np.log(rms_safe), np.nan)
    log_gain_pred = float(intercept) + float(slope) * log_seismic_rms
    log_gain_pred_clipped = np.clip(log_gain_pred, float(log_gain_clip[0]), float(log_gain_clip[1]))
    return pd.DataFrame(
        {
            "seismic_rms": seismic_rms,
            "log_seismic_rms": log_seismic_rms,
            "log_gain_pred": log_gain_pred,
            "log_gain_pred_clipped": log_gain_pred_clipped,
            "gain_pred": np.exp(log_gain_pred),
            "gain_pred_clipped": np.exp(log_gain_pred_clipped),
        }
    )


In [ ]:
batch_metrics_df = pd.read_csv(batch_metrics_file)
required_metric_columns = {
    "well_name",
    "status",
    "scale",
    "synthetic_qc_path",
    "depth_shift_curve_path",
    "corr",
    "nmae",
}
missing = required_metric_columns - set(batch_metrics_df.columns)
if missing:
    raise ValueError(f"Batch metrics missing columns: {sorted(missing)}")

ok_batch_df = batch_metrics_df.loc[batch_metrics_df["status"] == "ok"].copy()
if min_batch_corr is not None:
    ok_batch_df = ok_batch_df.loc[ok_batch_df["corr"] >= float(min_batch_corr)].copy()
if ok_batch_df.empty:
    raise ValueError("No successful batch synthetic wells available for fitting/QC.")

segment_rows = []
for _, row in ok_batch_df.iterrows():
    well_name = str(row["well_name"])
    scale = float(row["scale"])
    if not np.isfinite(scale) or scale <= 0.0:
        continue

    qc_path = resolve_artifact_path(row["synthetic_qc_path"])
    depth_shift_curve_path = resolve_artifact_path(row["depth_shift_curve_path"])
    if not qc_path.exists():
        raise FileNotFoundError(qc_path)
    if not depth_shift_curve_path.exists():
        raise FileNotFoundError(depth_shift_curve_path)

    qc_df = pd.read_csv(qc_path)
    required_qc_columns = {"twt_s", "seismic_norm", "synthetic_scaled", "eval_mask"}
    missing = required_qc_columns - set(qc_df.columns)
    if missing:
        raise ValueError(f"Synthetic QC {qc_path} missing columns: {sorted(missing)}")

    depth_shift_df = pd.read_csv(depth_shift_curve_path)
    required_depth_columns = {"twt_s", "tvdss_m"}
    missing = required_depth_columns - set(depth_shift_df.columns)
    if missing:
        raise ValueError(f"Depth shift curve {depth_shift_curve_path} missing columns: {sorted(missing)}")

    twt_s = qc_df["twt_s"].to_numpy(dtype=float)
    seismic_norm = qc_df["seismic_norm"].to_numpy(dtype=float)
    synthetic_scaled = qc_df["synthetic_scaled"].to_numpy(dtype=float)
    synthetic_raw = synthetic_scaled / scale
    eval_mask = eval_mask_to_bool(qc_df["eval_mask"])

    depth_twt = depth_shift_df["twt_s"].to_numpy(dtype=float)
    depth_z = depth_shift_df["tvdss_m"].to_numpy(dtype=float)
    finite_depth = np.isfinite(depth_twt) & np.isfinite(depth_z)
    if int(finite_depth.sum()) < 2:
        raise ValueError(f"Depth shift curve {depth_shift_curve_path} has too few finite samples.")
    order = np.argsort(depth_twt[finite_depth])
    depth_twt = depth_twt[finite_depth][order]
    depth_z = depth_z[finite_depth][order]

    finite = eval_mask & np.isfinite(twt_s) & np.isfinite(seismic_norm) & np.isfinite(synthetic_raw)
    valid_indices = np.flatnonzero(finite)
    segments = split_valid_indices(
        valid_indices,
        min_valid_samples=min_segment_valid_samples,
        max_segments=max_segment_count,
        min_segments=min_segments_per_well,
    )

    for segment_id, segment_indices in enumerate(segments):
        seg_twt = twt_s[segment_indices]
        seg_seismic = seismic_norm[segment_indices]
        seg_synthetic_raw = synthetic_raw[segment_indices]
        gain = positive_ls_gain(
            seg_seismic,
            seg_synthetic_raw,
            eps=gain_eps,
            min_valid_samples=min_segment_valid_samples,
        )
        if not np.isfinite(gain):
            continue

        attrs = segment_attribute_values(seg_seismic)
        tvdss_values = np.interp(seg_twt, depth_twt, depth_z)
        segment_rows.append(
            {
                "well_name": well_name,
                "segment_id": int(segment_id),
                "n_valid_samples": int(segment_indices.size),
                "twt_min_s": float(np.nanmin(seg_twt)),
                "twt_max_s": float(np.nanmax(seg_twt)),
                "tvdss_min_m": float(np.nanmin(tvdss_values)),
                "tvdss_max_m": float(np.nanmax(tvdss_values)),
                "segment_thickness_m": float(np.nanmax(tvdss_values) - np.nanmin(tvdss_values)),
                "gain": gain,
                "batch_scale": scale,
                "batch_corr": float(row["corr"]),
                **attrs,
            }
        )

training_samples_df = pd.DataFrame(segment_rows)
if training_samples_df.empty:
    raise ValueError("No finite training segments were generated from batch synthetic outputs.")

for column in ["gain", "seismic_rms", "seismic_abs_mean", "seismic_abs_p90"]:
    set_log_column(training_samples_df, column, f"log_{column}")
training_samples_df.to_csv(training_samples_file, index=False)

required_columns = {"log_gain", f"log_{selected_attribute}", "gain", selected_attribute, "segment_thickness_m"}
missing = required_columns - set(training_samples_df.columns)
if missing:
    raise ValueError(f"Training samples missing columns: {sorted(missing)}")

fit_df = training_samples_df.loc[
    np.isfinite(training_samples_df["log_gain"]) & np.isfinite(training_samples_df[f"log_{selected_attribute}"])
].copy()
if fit_df.empty:
    raise ValueError(f"No finite ln(gain) / ln({selected_attribute}) samples.")

fit = fit_log_linear(
    fit_df[f"log_{selected_attribute}"].to_numpy(dtype=float),
    fit_df["log_gain"].to_numpy(dtype=float),
)
log_gain_clip = tuple(np.nanpercentile(fit_df["log_gain"].to_numpy(dtype=float), prediction_clip_percentiles))
attribute_floor = float(
    np.nanpercentile(fit_df[selected_attribute].to_numpy(dtype=float), 1.0) * attribute_floor_fraction
)
attribute_floor = max(attribute_floor, np.finfo(float).tiny)

segment_thickness_m = fit_df["segment_thickness_m"].to_numpy(dtype=float)
segment_thickness_m = segment_thickness_m[np.isfinite(segment_thickness_m) & (segment_thickness_m > 0.0)]
if application_rms_window_m is None:
    application_window_m = float(np.nanmedian(segment_thickness_m))
else:
    application_window_m = float(application_rms_window_m)

fit_metrics_df = pd.DataFrame(
    [
        {
            **fit,
            "target": "log_gain",
            "attribute": f"log_{selected_attribute}",
            "n_training_segments": int(fit_df.shape[0]),
            "n_training_wells": int(fit_df["well_name"].nunique()),
            "log_gain_clip_p05": float(log_gain_clip[0]),
            "log_gain_clip_p95": float(log_gain_clip[1]),
            "gain_clip_p05": float(np.exp(log_gain_clip[0])),
            "gain_clip_p95": float(np.exp(log_gain_clip[1])),
            "attribute_floor": attribute_floor,
            "application_window_m": application_window_m,
            "min_segment_valid_samples": int(min_segment_valid_samples),
            "max_segment_count": int(max_segment_count),
            "min_segments_per_well": int(min_segments_per_well),
            "batch_metrics_file": str(batch_metrics_file),
            "training_samples_file": str(training_samples_file),
        }
    ]
)
fit_metrics_df.to_csv(fit_metrics_file, index=False)

fit_row = fit_metrics_df.iloc[0]
print(f"ln(gain) = {fit['intercept']:.3f} + {fit['slope']:.3f} * ln({selected_attribute})")
print("Training dynamic segmentation:")
print("  min_segment_valid_samples =", min_segment_valid_samples)
print("  max_segment_count =", max_segment_count)
print("  min_segments_per_well =", min_segments_per_well)
print("Fit samples:", int(fit_row["n_training_segments"]), "segments from", int(fit_row["n_training_wells"]), "wells")
print("Fit R2:", float(fit_row["r2_in_sample"]), "pearson:", float(fit_row["pearson"]))
print("Gain clip:", float(fit_row["gain_clip_p05"]), "to", float(fit_row["gain_clip_p95"]))
print("Application RMS window (m):", application_window_m)
print("Saved", training_samples_file)
print("Saved", fit_metrics_file)
display(fit_metrics_df)


In [ ]:
print("Wells for well-side QC:", ok_batch_df["well_name"].tolist())
display(ok_batch_df[["well_name", "corr", "nmae", "scale", "synthetic_qc_path", "depth_shift_curve_path"]])


In [ ]:
summary_rows = []
well_frames = []

for _, row in ok_batch_df.iterrows():
    well_name = str(row["well_name"])
    name = safe_name(well_name)
    scale = float(row["scale"])
    if not np.isfinite(scale) or scale <= 0.0:
        print("Skipping", well_name, "because batch scale is invalid:", scale)
        continue

    qc_path = resolve_artifact_path(row["synthetic_qc_path"])
    depth_shift_curve_path = resolve_artifact_path(row["depth_shift_curve_path"])
    if not qc_path.exists():
        raise FileNotFoundError(qc_path)
    if not depth_shift_curve_path.exists():
        raise FileNotFoundError(depth_shift_curve_path)

    qc_df = pd.read_csv(qc_path)
    required_qc_columns = {"twt_s", "seismic_norm", "reflectivity_shifted", "synthetic_scaled", "eval_mask"}
    missing = required_qc_columns - set(qc_df.columns)
    if missing:
        raise ValueError(f"Synthetic QC {qc_path} missing columns: {sorted(missing)}")

    depth_shift_df = pd.read_csv(depth_shift_curve_path)
    required_depth_columns = {"twt_s", "tvdss_m"}
    missing = required_depth_columns - set(depth_shift_df.columns)
    if missing:
        raise ValueError(f"Depth shift curve {depth_shift_curve_path} missing columns: {sorted(missing)}")

    twt_s = qc_df["twt_s"].to_numpy(dtype=float)
    seismic_norm = qc_df["seismic_norm"].to_numpy(dtype=float)
    synthetic_fixed_scale = qc_df["synthetic_scaled"].to_numpy(dtype=float)
    synthetic_raw = synthetic_fixed_scale / scale
    eval_mask = eval_mask_to_bool(qc_df["eval_mask"])

    depth_twt = depth_shift_df["twt_s"].to_numpy(dtype=float)
    depth_z = depth_shift_df["tvdss_m"].to_numpy(dtype=float)
    finite_depth = np.isfinite(depth_twt) & np.isfinite(depth_z)
    if int(finite_depth.sum()) < 2:
        raise ValueError(f"Depth shift curve {depth_shift_curve_path} has too few finite samples.")
    order = np.argsort(depth_twt[finite_depth])
    depth_twt = depth_twt[finite_depth][order]
    depth_z = depth_z[finite_depth][order]
    tvdss_m = np.interp(twt_s, depth_twt, depth_z, left=np.nan, right=np.nan)

    window_samples, depth_step_m = resolve_odd_window_samples(application_window_m, tvdss_m)
    pred_df = predict_gain_from_trace_rms(
        seismic_norm,
        window_samples=window_samples,
        intercept=fit["intercept"],
        slope=fit["slope"],
        attribute_floor=attribute_floor,
        log_gain_clip=log_gain_clip,
    )
    gain_curve = pred_df[prediction_gain_column].to_numpy(dtype=float)
    synthetic_gain_pred = gain_curve * synthetic_raw

    fixed_metrics = candidate_metrics(seismic_norm, synthetic_fixed_scale, eval_mask)
    gain_metrics = candidate_metrics(seismic_norm, synthetic_gain_pred, eval_mask)

    out_df = pd.DataFrame(
        {
            "well_name": well_name,
            "twt_s": twt_s,
            "tvdss_m": tvdss_m,
            "seismic_norm": seismic_norm,
            "reflectivity_shifted": qc_df["reflectivity_shifted"].to_numpy(dtype=float),
            "synthetic_raw": synthetic_raw,
            "synthetic_fixed_scale": synthetic_fixed_scale,
            "synthetic_gain_pred": synthetic_gain_pred,
            "residual_fixed_scale": seismic_norm - synthetic_fixed_scale,
            "residual_gain_pred": seismic_norm - synthetic_gain_pred,
            "eval_mask": eval_mask,
            "batch_scale": scale,
            "rms_window_samples": window_samples,
            "rms_window_m": window_samples * depth_step_m,
        }
    )
    out_df = pd.concat([out_df, pred_df], axis=1)

    out_path = well_qc_dir / f"wellside_gain_synthetic_qc_{name}.csv"
    out_df.to_csv(out_path, index=False)
    well_frames.append(out_df)

    finite_gain = gain_curve[np.isfinite(gain_curve)]
    summary_rows.append(
        {
            "well_name": well_name,
            "n_samples": int(twt_s.size),
            "n_eval_samples": int(eval_mask.sum()),
            "batch_corr": float(row["corr"]),
            "batch_nmae": float(row["nmae"]),
            "fixed_corr_recomputed": fixed_metrics["corr"],
            "fixed_nmae_recomputed": fixed_metrics["nmae"],
            "gain_corr": gain_metrics["corr"],
            "gain_nmae": gain_metrics["nmae"],
            "batch_scale": scale,
            "gain_median": float(np.nanmedian(finite_gain)) if finite_gain.size else np.nan,
            "gain_p10": float(np.nanpercentile(finite_gain, 10.0)) if finite_gain.size else np.nan,
            "gain_p90": float(np.nanpercentile(finite_gain, 90.0)) if finite_gain.size else np.nan,
            "rms_window_samples": int(window_samples),
            "inferred_depth_step_m": depth_step_m,
            "rms_window_m": float(window_samples * depth_step_m),
            "well_qc_path": str(out_path),
        }
    )

summary_df = pd.DataFrame(summary_rows)
summary_path = output_dir / "wellside_gain_synthetic_qc_summary.csv"
summary_df.to_csv(summary_path, index=False)
print("Saved", summary_path)
display(summary_df)


In [ ]:
for _, summary in summary_df.iterrows():
    well_name = str(summary["well_name"])
    name = safe_name(well_name)
    qc_path = Path(summary["well_qc_path"])
    qc_df = pd.read_csv(qc_path)

    depth = qc_df["tvdss_m"].to_numpy(dtype=float)
    seismic = qc_df["seismic_norm"].to_numpy(dtype=float)
    fixed_syn = qc_df["synthetic_fixed_scale"].to_numpy(dtype=float)
    gain_syn = qc_df["synthetic_gain_pred"].to_numpy(dtype=float)
    gain = qc_df[prediction_gain_column].to_numpy(dtype=float)
    rms = qc_df["seismic_rms"].to_numpy(dtype=float)

    fig, axes = plt.subplots(1, 3, figsize=(12.5, 5.4), sharey=True, constrained_layout=True)

    axes[0].plot(seismic, depth, lw=0.9, color="black", label="Seismic")
    axes[0].plot(fixed_syn, depth, lw=0.9, color="tab:red", alpha=0.85, label="Fixed-scale synthetic")
    axes[0].invert_yaxis()
    axes[0].set_xlabel("Normalized amplitude")
    axes[0].set_ylabel("TVDSS (m)")
    axes[0].set_title(
        f"Fixed: corr={summary['fixed_corr_recomputed']:.3f}, nmae={summary['fixed_nmae_recomputed']:.3f}"
    )
    axes[0].grid(True, alpha=0.25)
    axes[0].legend(loc="best", fontsize=7)

    axes[1].plot(seismic, depth, lw=0.9, color="black", label="Seismic")
    axes[1].plot(gain_syn, depth, lw=0.9, color="tab:blue", alpha=0.9, label="RMS-gain synthetic")
    axes[1].set_xlabel("Normalized amplitude")
    axes[1].set_title(f"Gain: corr={summary['gain_corr']:.3f}, nmae={summary['gain_nmae']:.3f}")
    axes[1].grid(True, alpha=0.25)
    axes[1].legend(loc="best", fontsize=7)

    gain_norm = gain / np.nanmedian(gain[np.isfinite(gain)]) if np.any(np.isfinite(gain)) else gain
    rms_norm = rms / np.nanmedian(rms[np.isfinite(rms)]) if np.any(np.isfinite(rms)) else rms
    axes[2].plot(gain_norm, depth, lw=1.0, color="tab:blue", label="gain / median")
    axes[2].plot(rms_norm, depth, lw=1.0, color="tab:green", label="RMS / median")
    axes[2].set_xlabel("Normalized attribute")
    axes[2].set_title("Predicted gain curve")
    axes[2].grid(True, alpha=0.25)
    axes[2].legend(loc="best", fontsize=7)

    fig.suptitle(well_name)
    fig_path = figure_dir / f"qc_{name}_wellside_gain_synthetic.png"
    fig.savefig(fig_path, dpi=180, bbox_inches="tight")
    plt.show()
    print("Saved", fig_path)


In [ ]:
if summary_df.empty:
    raise ValueError("No well-side QC rows to summarize.")

labels = summary_df["well_name"].tolist()
x = np.arange(len(summary_df))
width = 0.34

fig, axes = plt.subplots(1, 3, figsize=(15, 4.3), constrained_layout=True)

axes[0].bar(x - width / 2, summary_df["fixed_corr_recomputed"], width=width, color="tab:red", alpha=0.8, label="fixed")
axes[0].bar(x + width / 2, summary_df["gain_corr"], width=width, color="tab:blue", alpha=0.8, label="gain")
axes[0].set_xticks(x)
axes[0].set_xticklabels(labels, rotation=45, ha="right")
axes[0].set_ylabel("Correlation")
axes[0].set_ylim(-1, 1)
axes[0].set_title("Synthetic correlation")
axes[0].grid(True, axis="y", alpha=0.25)
axes[0].legend(loc="best", fontsize=8)

axes[1].bar(x - width / 2, summary_df["fixed_nmae_recomputed"], width=width, color="tab:red", alpha=0.8, label="fixed")
axes[1].bar(x + width / 2, summary_df["gain_nmae"], width=width, color="tab:blue", alpha=0.8, label="gain")
axes[1].set_xticks(x)
axes[1].set_xticklabels(labels, rotation=45, ha="right")
axes[1].set_ylabel("NMAE")
axes[1].set_title("Synthetic NMAE")
axes[1].grid(True, axis="y", alpha=0.25)
axes[1].legend(loc="best", fontsize=8)

axes[2].plot(x, summary_df["batch_scale"], marker="o", lw=1.1, color="tab:red", label="batch fixed scale")
axes[2].plot(x, summary_df["gain_median"], marker="o", lw=1.1, color="tab:blue", label="median predicted gain")
axes[2].fill_between(
    x,
    summary_df["gain_p10"].to_numpy(dtype=float),
    summary_df["gain_p90"].to_numpy(dtype=float),
    color="tab:blue",
    alpha=0.15,
    label="gain P10-P90",
)
axes[2].set_xticks(x)
axes[2].set_xticklabels(labels, rotation=45, ha="right")
axes[2].set_ylabel("Gain")
axes[2].set_title("Fixed scale vs predicted gain")
axes[2].grid(True, axis="y", alpha=0.25)
axes[2].legend(loc="best", fontsize=8)

summary_fig_path = figure_dir / "qc_00_wellside_gain_synthetic_summary.png"
fig.savefig(summary_fig_path, dpi=180, bbox_inches="tight")
plt.show()
print("Saved", summary_fig_path)


In [ ]:
print("Summary:")
print(summary_path)
print()
print("Per-well QC CSV files:")
for path in sorted(well_qc_dir.glob("*.csv")):
    print(path)
print()
print("Figures:")
for path in sorted(figure_dir.glob("*.png")):
    print(path)
